# Data Cleaning with Pandas

In this notebook we'll go through a few basic data cleaning steps that should be performed on all new datasets where necessary.

We'll go through the process with both the `orders` and `orderlines` datasets. You can then practice these skills by cleaning the `products` dataset yourself

In [19]:
import pandas as pd

In [20]:
# orders.csv
url = "https://drive.google.com/file/d/1e3z9GBBwRbwa5s09Tv2H8J6BgwfTxzK8/view?usp=sharing"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orders = pd.read_csv(path)

# orderlines.csv
url = "https://drive.google.com/file/d/12FXbgkMipb-vKab258ykTcnWMebPH0pb/view?usp=sharing"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orderlines = pd.read_csv(path)

In [21]:
# brands.csv
url = "https://drive.google.com/file/d/14ER-Jcvfu1WYgzMClaBX7itOZ9U91gwk/view?usp=sharing"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
brands = pd.read_csv(path)

Before we begin, let's create a copy of the `orders` and `orderlines` DataFrames. This way we are sure any of our changes won't affect the original DataFrames

In [22]:
orders_df = orders.copy()

In [23]:
orderlines_df = orderlines.copy()

In [24]:
brands_df = brands.copy()

One of the best ways to begin data cleaning is by exploring using `.info()`. This will tell us:
* The shape of the DataFrame
* The names of the columns
* If there are any missing values
* The datatypes of the columns

By exploring the missing values and correcting any incorrect datatypes, we often come across inconsistencies in our data.

Beyond this, we should also have a **check for any duplicate rows**.

Let's first deal with the duplicates, as it's nice and easy, then we'll explore what `.info()` has to tell us.

## 1.&nbsp; Duplicates
We can check for duplicates using the pandas [.duplicated()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html) method.

We can then delete these rows, if we wish, using [.drop_duplicates()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html)

In [25]:
# orders_df
orders_df.duplicated().sum()

np.int64(0)

In [26]:
# orderlines_df
orderlines_df.duplicated().sum()

np.int64(0)

We have no duplicate rows in either DataFrame. Easy, there is no problem to solve. Normally though, if there were some duplicates, we'd drop the extra rows.

We can also check if a single column contains duplicates. This is especially useful when there should be unique values in a column.

In [27]:
brands_df.duplicated("long").sum()

np.int64(6)

Some brands have 2 different abbreviations. Exploring the products of these brands you may see that MUJ and KEN have the wrong long brand name listed. We will therefore correct this by reassigning the right brand name. You can inspect the brands further by merging with the products dataframe.

In [28]:
brands.loc[(brands["long"].duplicated(keep=False))].sort_values(by="long")

,short,long
6,AP2,Apple
7,APP,Apple
17,BOS,Bose
19,CAD,Bose
67,JYB,Jaybird
70,KEN,Jaybird
94,MOP,Mophie
98,MUJ,Mophie
117,OTR,Startech
153,STA,Startech


In [29]:
brands.loc[brands["short"] == "MUJ", "long"] = "Mujjo"

In [30]:
brands.loc[brands["short"] == "KEN", "long"] = "Kensington"

In [31]:
brands.loc[(brands["long"].duplicated(keep=False))].sort_values(by="long")

,short,long
6,AP2,Apple
7,APP,Apple
17,BOS,Bose
19,CAD,Bose
117,OTR,Startech
153,STA,Startech
37,ENV,Unknown
80,LIB,Unknown


# 2.&nbsp; `.info()`

In [32]:
orders_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226909 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      226909 non-null  int64  
 1   created_date  226909 non-null  object 
 2   total_paid    226904 non-null  float64
 3   state         226909 non-null  object 
dtypes: float64(1), int64(1), object(2)
memory usage: 6.9+ MB


* `total_paid` has 5 missing values
* `created_date` should become datetime datatype

In [33]:
orderlines_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 293983 entries, 0 to 293982
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   id                293983 non-null  int64 
 1   id_order          293983 non-null  int64 
 2   product_id        293983 non-null  int64 
 3   product_quantity  293983 non-null  int64 
 4   sku               293983 non-null  object
 5   unit_price        293983 non-null  object
 6   date              293983 non-null  object
dtypes: int64(4), object(3)
memory usage: 15.7+ MB


* `date` should be a datetime datatype
* `unit_price` should be a float datatype

## 3.&nbsp; Missing values

### 3.1.&nbsp; Orders
* `total_paid` has 5 missing values

In [34]:
orders_df.loc[orders_df['total_paid'].isna()]

,order_id,created_date,total_paid,state
127701,427314,2017-11-20 18:54:39,NaN,Pending
132013,431655,2017-11-22 12:15:24,NaN,Pending
147316,447411,2017-11-27 10:32:37,NaN,Pending
148833,448966,2017-11-27 18:54:15,NaN,Pending
149434,449596,2017-11-27 21:52:08,NaN,Pending


In [35]:
num_missing = orders_df['total_paid'].isna().sum()
total_rows = orders_df.shape[0]
percent_missing = (100*num_missing/total_rows)
print(f"5 missing values represents {percent_missing:.5f}% of the rows in our DataFrame")

5 missing values represents 0.00220% of the rows in our DataFrame


> A quick way to find out a percentage if you don't need to print out a sentence for yourself/students/colleagues is `.value_counts(normalize=True)`

In [36]:
orders_df['total_paid'].isna().value_counts(normalize=True)

,proportion
total_paid,
False,0.999978
True,0.000022


As there is such a tiny amount of missing values, we will simply delete these rows, as we have enough data without them.

In [37]:
orders_df = orders_df.dropna(axis=0)

Should you have a significant number of missing values in the future, you have a choice:
+ you can impute the values
+ you can take the values from other DataFrames if they are redundantly stored
+ you can delete the rows or columns
+ or any number of other creative solutions

Please, always consider how much time you have on your project, and what impact your method of choice will have on your final assesment.

### 3.2.&nbsp; Orderlines
There are no missing values in `orderlines_df`

## 4.&nbsp; Datatypes

### 4.1.&nbsp; Orders
* `created_date` should become datetime datatype

In [38]:
orders_df["created_date"] = pd.to_datetime(orders_df["created_date"]).copy()

### 4.1.&nbsp; Orderlines
* `date` should be a datetime datatype
* `unit_price` should be a float datatype

#### 4.1.1.&nbsp; `date`

In [39]:
orderlines_df["date"] = pd.to_datetime(orderlines_df["date"]).copy()

#### 4.1.2.&nbsp;`unit_price`

In [40]:
orderlines_df["unit_price"] = pd.to_numeric(orderlines_df["unit_price"])

ValueError: Unable to parse string "1.137.99" at position 6

In [ ]:
pd.to_numeric(orderlines_df["unit_price"],errors="coerce").isna().sum()

np.int64(36169)

As you can see when we try to convert `unit_price` to a numerical datatype, we receive a `ValueError` telling us that pandas doesn't understand the number `1.137.99`. This is probably because numbers cannot have multiple decimal points. Let's see if there are any other numbers like this:

> `.` is a wildcard in regex, we need the `\` as an escape

In [41]:
# Count the number of decimal points in the unit_price
orderlines_df['unit_price'].str.count(r"\.").value_counts()

,count
unit_price,
1,257814
2,36169


Looks like over 36000 rows in `orderlines` are affected by this problem. Let's work out how much that is as a percentage of our total data.

In [42]:
# Count the rows with more than one `.`
mult_decimal_rows = (orderlines_df['unit_price'].str.count(r"\.")>1).sum()

# Find the percentage of corrupted rows
percent_corrupted = (100 * mult_decimal_rows / orderlines_df.shape[0])
print(f"{percent_corrupted:.2f}% of the rows in our DataFrame have multiple decimal points in the unit_price")

12.30% of the rows in our DataFrame have multiple decimal points in the unit_price


This is a bit of a tricky decision as 12.3% is a significant amount of our data... and we might even end up losing a larger portion of our data than this too. For the moment we will delete the rows as we only have 2 weeks for this project and I'd like some quick, accurate results to show. If we have time at the end, we can come back and investigate this problem further, maybe there's a solution?

Each row of `orderlines` represents a product in an order. For example, if order number 175 contained 3 seperate products, then order 175 would have 3 rows in `orderlines`, one row for each of the products. If 2 of those products have 'normal' prices (14.99, 15.85) and 1 has a price with 2 decimal points (1.137.99), we need to remove the whole order and not just the affected row. If we only remove the row with 2 decimal places then any later analysis about products and prices could be misleading.

We therefore need to find the order numbers associated with the rows that have 2 decimal points, and then remove all the associated rows.

In [43]:
# Boolean mask to find the orders that contain a price with multiple decimal points
multiple_decimal_mask = orderlines_df['unit_price'].str.count(r"\.") > 1

# Apply the boolean mask to the orderlines DataFrame. This way we can find the order_id of all the affected orders.
corrupted_order_ids = orderlines_df.loc[multiple_decimal_mask, "id_order"]

# Keep only the rows that do not have multiple decimal points
orderlines_df = orderlines_df.loc[~orderlines_df['id_order'].isin(corrupted_order_ids)]

In [44]:
orderlines_df.shape[0]

216250

In [ ]:
# alternative: get rid of first dot and leave the second as decimal point
# # orderlines: remove first '.' in unit prices with multiple periods (have weird format, e.g. 3.222.52)
# mult_decimal_mask = (orderlines_df['unit_price'].str.count(r"\.")>1)
# orderlines_df.loc[mult_decimal_mask,'unit_price'] = orderlines_df.loc[mult_decimal_mask,'unit_price'].str.replace('.', '', n=1)

We still have 216250 rows in orderlines to work with. This should be more than enough for our evaluation.

Now that all of the 2 decimal point prices have been removed, let's try again to convert the column `unit_price` to the correct datatype.

In [45]:
orderlines_df["unit_price"] = pd.to_numeric(orderlines_df["unit_price"])

In [46]:
orderlines_df

,id,id_order,product_id,product_quantity,sku,unit_price,date
0,1119109,299539,0,1,OTT0133,18.99,2017-01-01 00:07:19
1,1119110,299540,0,1,LGE0043,399.00,2017-01-01 00:19:45
2,1119111,299541,0,1,PAR0071,474.05,2017-01-01 00:20:57
3,1119112,299542,0,1,WDT0315,68.39,2017-01-01 00:51:40
4,1119113,299543,0,1,JBL0104,23.74,2017-01-01 01:06:38
...,...,...,...,...,...,...,...
293978,1650199,527398,0,1,JBL0122,42.99,2018-03-14 13:57:25
293979,1650200,527399,0,1,PAC0653,141.58,2018-03-14 13:57:34
293980,1650201,527400,0,2,APP0698,9.99,2018-03-14 13:57:41
293981,1650202,527388,0,1,BEZ0204,19.99,2018-03-14 13:58:01


It worked perfectly

# Challenge: Clean the `products` DataFrame
Now it's your turn. Use the lessons you learnt above and clean the products DataFrame. You don't have to copy exactly what we did. Think about the consequences of your actions, sometimes it is ok to delete rows, other times you may wish to come up with more creative solutions.

In [47]:
# products.csv
url = "https://drive.google.com/file/d/12kxuogonsJHR_V3cMsEVEQBaAWdz9zie/view?usp=sharing"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
products = pd.read_csv(path)

In [48]:
# your code here
products_df = products.copy()

In [49]:
products_df.sample(5)

,sku,name,desc,price,promo_price,in_stock,type
13151,BEL0269,Belkin RockStar MIXIT Power Bank 10000 mAh Ext...,Portable external battery compartment for cabl...,99.99,69.99,0,1515
11714,ADN0040,Adonith Mark Stylus Black,digital pen with standard anti-roll point and ...,12.99,128.998,0,1229
278,PAC0184,"Apple MacBook Pro 133 ""25GHz | RAM 16GB | 500G...",Apple MacBook Pro 133 inches (MD101Y / A) with...,1439,13.989.899,0,1282
10671,LUN0010,Epik lunatik Correa and housing 42mm Apple Wat...,Polycarbonate shell and silicone strap Apple W...,59.95,499.899,0,2434
4134,APP1389,"Apple iMac 27 ""Core i5 3.3GHz Retina 5K | 8GB ...",IMac desktop computer 27 inch 8GB RAM 512GB Re...,3169,30.175.839,0,"5,74E+15"


In [50]:
products_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19326 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   sku          19326 non-null  object
 1   name         19326 non-null  object
 2   desc         19319 non-null  object
 3   price        19280 non-null  object
 4   promo_price  19326 non-null  object
 5   in_stock     19326 non-null  int64 
 6   type         19276 non-null  object
dtypes: int64(1), object(6)
memory usage: 1.0+ MB


We'll go through the steps above in order

* Duplicates
* Missing values
* Datatypes

But I think we can all see straight away from **products.head()** above that some of the prices in **promo_price** look wrong. Let's make sure we deal with this later.



### Look for Duplicates

In [51]:
# your code here
products_df.duplicated().sum()

np.int64(8746)

Wow, that's a lot of duplicates.Let's get rid of them.

In [52]:
products_df = products_df.drop_duplicates().copy()

In [53]:
products_df

,sku,name,desc,price,promo_price,in_stock,type
0,RAI0007,Silver Rain Design mStand Support,Aluminum support compatible with all MacBook,59.99,499.899,1,8696
1,APP0023,Apple Mac Keyboard Keypad Spanish,USB ultrathin keyboard Apple Mac Spanish.,59,589.996,0,13855401
2,APP0025,Mighty Mouse Apple Mouse for Mac,mouse Apple USB cable.,59,569.898,0,1387
3,APP0072,Apple Dock to USB Cable iPhone and iPod white,IPhone dock and USB Cable Apple iPod.,25,229.997,0,1230
4,KIN0007,Mac Memory Kingston 2GB 667MHz DDR2 SO-DIMM,2GB RAM Mac mini and iMac (2006/07) MacBook Pr...,34.99,31.99,1,1364
...,...,...,...,...,...,...,...
19321,BEL0376,Belkin Travel Support Apple Watch Black,compact and portable stand vertically or horiz...,29.99,269.903,1,12282
19322,THU0060,"Enroute Thule 14L Backpack MacBook 13 ""Black",Backpack with capacity of 14 liter compartment...,69.95,649.903,1,1392
19323,THU0061,"Enroute Thule 14L Backpack MacBook 13 ""Blue",Backpack with capacity of 14 liter compartment...,69.95,649.903,1,1392
19324,THU0062,"Enroute Thule 14L Backpack MacBook 13 ""Red",Backpack with capacity of 14 liter compartment...,69.95,649.903,0,1392


In [54]:
products_df["sku"].duplicated().sum()
# only 1 sku is duplicated because it has a NaN value in the price column in one of the duplicates

np.int64(1)

In [55]:
products_df.loc[products_df["sku"].duplicated(keep = False)]

,sku,name,desc,price,promo_price,in_stock,type
7992,APP1197,"Apple iMac 21.5 ""Core i5 31 GHz Retina display...",Desktop Apple iMac 21.5 inch i5 31 GHz Retina ...,1729,1305.59,0,1282
8000,APP1197,"Apple iMac 21.5 ""Core i5 31 GHz Retina display...",Desktop Apple iMac 21.5 inch i5 31 GHz Retina ...,NaN,1305.59,0,1282


In [57]:
products_df.drop_duplicates(subset='sku', keep="first", inplace=True)
# why we keep first one(because the second one has NaN value in price)

In [58]:
products_df.loc[products_df["sku"] == "APP1197"]

,sku,name,desc,price,promo_price,in_stock,type
7992,APP1197,"Apple iMac 21.5 ""Core i5 31 GHz Retina display...",Desktop Apple iMac 21.5 inch i5 31 GHz Retina ...,1729,1305.59,0,1282


In [59]:
products_df["sku"].duplicated().sum()

np.int64(0)

### Look for Missing values


In [60]:
# your code here
products_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10579 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   sku          10579 non-null  object
 1   name         10579 non-null  object
 2   desc         10572 non-null  object
 3   price        10534 non-null  object
 4   promo_price  10579 non-null  object
 5   in_stock     10579 non-null  int64 
 6   type         10529 non-null  object
dtypes: int64(1), object(6)
memory usage: 661.2+ KB




*  **desc** has 7 missing values

*  **price** has 46 missing values

*  **type** has 50 missing values

* **price** should be float datatype
* **promo_price** should be float datatype




#####**desc**

In [61]:
products_df["desc"].isna().sum()

np.int64(7)

In [62]:
products_df.loc[products_df['desc'].isna(),:]

,sku,name,desc,price,promo_price,in_stock,type
16126,WDT0211-A,"Open - Purple 2TB WD 35 ""PC Security Mac hard ...",NaN,107,814.659,0,1298
16128,APP1622-A,Open - Apple Smart Keyboard Pro Keyboard Folio...,NaN,1.568.206,1.568.206,0,1298
17843,PAC2334,Synology DS718 + NAS Server | 10GB RAM,NaN,566.35,5.659.896,0,12175397
18152,KAN0034-A,Open - Kanex USB-C Gigabit Ethernet Adapter Ma...,NaN,29.99,237.925,0,1298
18490,HTE0025,Hyper Pearl 1600mAh battery Mini USB Mirror an...,NaN,24.99,22.99,1,1515
18612,OTT0200,OtterBox External Battery Power Pack 20000 mAHr,NaN,79.99,56.99,1,1515
18690,HOW0001-A,Open - Honeywell thermostat Lyric zonificador ...,NaN,199.99,1.441.174,0,11905404


We have 2 choices here:

* We can quickly and easily remove these rows.
* Or, alternatively, the products names here are quite descriptive, so I'm tempted to just copy them to the description column, so that there is a description if we later want utilise this column. I wouldn't recommend this if this DataFrame was the source of truth for our website. But this is not the case here, and we're not faking any information (guessing a price or so), so I'm happy with this option

In [63]:
products_df.loc[products_df['desc'].isna(),'desc'] = products_df.loc[products_df['desc'].isna(),'name']

In [64]:
products_df.desc.isna().sum()

np.int64(0)

### Check / Change Data types

####**price**

In [65]:
# your code here
products_df.price.isna().sum()

np.int64(45)

In [66]:
num_missing = products_df['price'].isna().sum()
total_rows = products_df.shape[0]
percent_missing = (100*num_missing/total_rows)
print(f"45 missing values in price represents { percent_missing:.2f}% of the rows in our Dataframe")


45 missing values in price represents 0.43% of the rows in our Dataframe


Let's simply delete these rows to ensure that we can trust the numbers in our final DataFrame. Afterall, the price is very important when investigating discounts.

In [67]:
products_df = products_df.loc[~products['price'].isna()] # option1 .loc method
# products_df = products_df.dropna(subset=['price']) # option2. dropna() method

In [68]:
products_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10534 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   sku          10534 non-null  object
 1   name         10534 non-null  object
 2   desc         10534 non-null  object
 3   price        10534 non-null  object
 4   promo_price  10534 non-null  object
 5   in_stock     10534 non-null  int64 
 6   type         10484 non-null  object
dtypes: int64(1), object(6)
memory usage: 658.4+ KB


Type isn’t an essential piece of data for the analysis and is therefore allowed to carry missing values. The only place it comes in later is as an optional route to category creation, where someone might still choose to drop the rows with missing values, however one can still use name and desc to categorize those rows.

**Data types**

We saw from looking at the output of .info() that both price and promo_price have been stored as objects and not as a numerical datatypes. We also saw while solving other problems that both columns have some prices with 3 decimal places and others with 2 decimal points - the latter will prevent us from converting the datatype to numerical, so first we must investigate and solve these problems.

**price**

In [69]:
#First, let's see how many values are affected by the 2-decimal-dot problems or 3 decimal places.

products_df.loc[(products_df["price"].str.contains(r"\d+\.\d+\.\d+"))]

,sku,name,desc,price,promo_price,in_stock,type
665,CRU0015-2,Crucial memory Mac 16GB (2x8GB) SO-DIMM DDR3 1...,RAM 16GB (2x8GB) 135V MacBook Pro iMac (2012/2...,1.639.792,1.629.894,1,1364
792,APP0672,Apple iPhone 5S 16GB Space Gray,New iPhone 5S 16G Libre (ME432Y / AB).,4.694.994,4.694.994,0,NaN
797,APP0673,Apple iPhone 5S 16GB Silver,New Free iPhone 5S 16GB (ME433Y / A).,4.090.042,4.090.042,0,NaN
827,PAC0339,NewerTech miniStack 4TB Hard Drive Mac,External Box Hard Drive Mac + 4TB.,2.199.791,2.199.901,0,11935397
885,PAC0376,OWC Mercury Elite Pro Dual Thunderbolt + 8TB,RAID outer box 35 inch SATA connection Thunder...,5.609.698,5.549.895,0,11935397
...,...,...,...,...,...,...,...
19312,REP0424,Input repair Headphones iPad,Repair service including parts and labor for iPad,6.999.003,69.99,0,"1,44E+11"
19313,REP0421,iPad charging connector repair,Repair service including parts and labor for iPad,6.999.003,69.99,0,"1,44E+11"
19314,REP0416,iPad front camera repair,Repair service including parts and labor for iPad,6.999.003,69.99,0,"1,44E+11"
19315,REP0413,repair rear camera iPad,Repair service including parts and labor for iPad,6.999.003,69.99,0,"1,44E+11"


In [70]:
# This code is a "diagnostic" filter used to find prices that have three or more digits after a decimal point (e.g., 1.299 or 1450.999).

products_df.loc[products_df["price"].str.contains(r"\d+\.\d{3,}")]

,sku,name,desc,price,promo_price,in_stock,type
362,REP0043,Speaker lower repair iPhone 4,Repair service including parts and labor for i...,499.004,499.004,0,"1,44E+11"
480,PIE0011,Internal Battery for iPhone 3G,Replacement AC Adapter for Apple iPhone 3G.,98.978,98.978,0,21485407
515,SEN0061,Sennheiser EZX 80 Handsfree iPhone iPad and iP...,IPhone bluetooth headset with microphone iPad ...,649.891,649.891,0,5384
518,SEV0026,Service installation RAM + HDD + SSD MacBook /...,RAM + HDD installation + SSD in your MacBook /...,599.918,599.918,0,20642062
525,SEV0024,Service installation RAM + HDD + SSD Mac mini,installation RAM HDD + SSD + on your Mac mini ...,599.918,599.918,0,20642062
...,...,...,...,...,...,...,...
19312,REP0424,Input repair Headphones iPad,Repair service including parts and labor for iPad,6.999.003,69.99,0,"1,44E+11"
19313,REP0421,iPad charging connector repair,Repair service including parts and labor for iPad,6.999.003,69.99,0,"1,44E+11"
19314,REP0416,iPad front camera repair,Repair service including parts and labor for iPad,6.999.003,69.99,0,"1,44E+11"
19315,REP0413,repair rear camera iPad,Repair service including parts and labor for iPad,6.999.003,69.99,0,"1,44E+11"


In [71]:
price_problems_number = products_df.loc[(products_df["price"].str.contains(r"\d+\.\d+\.\d+")) | (products_df["price"].str.contains(r"\d+\.\d{3,}")),:].shape[0]
price_problems_number

542

In [72]:
print(f"The column price has in total {price_problems_number} wrong values.This is {round(((price_problems_number/products_df.shape[0]) * 100),2)}% of the rows of the DataFrame")

The column price has in total 542 wrong values.This is 5.15% of the rows of the DataFrame


5.15% is a reasonable amount of our data. However, the price column will be important to understanding discounts, so I'd like it to be very trustworthy as we are basing business decisions on it. Therefore, we'll delete these rows.

In [73]:
products_df = products_df.loc[~((products_df.price.str.contains(r"\d+\.\d+\.\d+"))|(products_df.price.str.contains(r"\d+\.\d{3,}"))), :]

In [74]:
products_df.sample(5)

,sku,name,desc,price,promo_price,in_stock,type
17767,PAC2399,DS418play Synology NAS Server | 6GB RAM | 40TB...,4-bay NAS server to accommodate 4K Ultra HD files,2156.94,16.473.678,0,12175397
13370,APP1643,Apple iPhone 7 Plus 32GB Rose Gold,New Apple iPhone 32GB Rose Gold Plus 7 Free,779,7.640.001,0,85651716
1018,GTE0027,G-Tech G-RAID mini USB 3.0 1TB RAID Drive,portable disk RAID G-Technology FireWire 800 a...,294,2.289.901,0,11935397
1563,CRU0026-2,Crucial memory Mac 16GB (2x8GB) SO-DIMM DDR3 1...,RAM 16GB (2x8GB) Mac mini (2011) iMac (2010/11...,163.98,1.629.894,1,1364
18899,APP2370,"Apple Macbook Pro 13 ""Core i7 25GHz | 16GB | 1...",New MacBook Pro 13-inch Core i7 25 GHz with 16...,2099,19.730.042,0,"1,02E+12"


In [75]:
# To complete our task, let's convert the column to a numeric datatype

products_df["price"] = pd.to_numeric(products_df["price"])

In [76]:
products_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9992 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   sku          9992 non-null   object 
 1   name         9992 non-null   object 
 2   desc         9992 non-null   object 
 3   price        9992 non-null   float64
 4   promo_price  9992 non-null   object 
 5   in_stock     9992 non-null   int64  
 6   type         9946 non-null   object 
dtypes: float64(1), int64(1), object(5)
memory usage: 624.5+ KB


#### `promo_price`

Again, let's begin by seeing how many values are affected by the 2-decimal-dots problem, or the 3 decimal-places problem

In [77]:
promo_problems_number = products_df.loc[(products_df.promo_price.str.contains(r"\d+\.\d+\.\d+"))|(products_df.promo_price.str.contains(r"\d+\.\d{3,}")), :].shape[0]
promo_problems_number

9232

In [78]:
print(f"The column promo_price has in total {promo_problems_number} wrong values. This is {round(((promo_problems_number / products_df.shape[0]) * 100), 2)}% of the rows of the DataFrame")

The column promo_price has in total 9232 wrong values. This is 92.39% of the rows of the DataFrame


WOW!!! That's a lot of wrong data. Let's have a quick investigate to check that's correct. We'll make a DataFrame by copy-pasting the code we used above and then look at a large sample to check that all the numbers in the `promo_price` column really have either 2 decimal points or 3 decimal places.

In [79]:
promo_price_df = products_df.loc[(products_df.promo_price.str.contains(r"\d+\.\d+\.\d+"))|(products_df.promo_price.str.contains(r"\d+\.\d{3,}")), :]
promo_price_df.sample(5)

,sku,name,desc,price,promo_price,in_stock,type
15156,APP1374-A,"Open - Apple iMac 27 ""Core i7 Retina 5K 4GHz |...",IMac desktop computer 27 inch 8GB RAM 512GB Re...,3169.00,28.631.723,0,1298
776,MAK0023,Maclocks BrandMe Customizable iPad Stand Exclu...,IPad support for customizable floor with your ...,489.99,4.899.895,0,1216
2659,KAN0047,"Kanex Cable USB-C to USB-A 3.0 MacBook 12 ""- 1.2M",USB USB cable-C-A 3.0 for MacBook 12 inches.,19.99,169.896,0,1325
1502,PAC1818,Pack Synology DS1515 + | WD 30TB Network,DS1515 + NAS 8GB of memory and 10TB hard drive...,1435.92,13.169.628,0,12175397
913,PAC0391,OWC Data Doubler Pack MacBook / Macbook Pro Black,Pack Replacement Superdrive for SSD / HDD + bo...,121.98,559.903,1,12755395


So we were correct, over 90% of the data in this column is corrupt. There's no point deleting all of these rows, then we would barely have a products table. Instead, as it's only this column that appears to be very untrustworthy, we will delete the column.

In [80]:
products_cl = products_df.drop(columns=["promo_price"])

In [81]:
products_cl.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9992 entries, 0 to 19325
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   sku       9992 non-null   object 
 1   name      9992 non-null   object 
 2   desc      9992 non-null   object 
 3   price     9992 non-null   float64
 4   in_stock  9992 non-null   int64  
 5   type      9946 non-null   object 
dtypes: float64(1), int64(1), object(4)
memory usage: 546.4+ KB


Obviously, there's now no need to convert `promo_price` to a numerical datatype

Don't forget to download/save your new DataFrames. Also, give them an obvious name, so that you know they are the cleaned version and not the original DataFrame.

In [83]:
orders_cl = orders_df
orderlines_cl = orderlines_df
brands_cl = brands_df

In [84]:
from google.colab import files

orders_df.to_csv("orders_cl.csv", index=False)
files.download("orders_cl.csv")

orderlines_df.to_csv("orderlines_cl.csv", index=False)
files.download("orderlines_cl.csv")

products_df.to_csv("products_cl.csv", index=False)
files.download("products_cl.csv")

brands_df.to_csv("brands_cl.csv", index=False)
files.download("brands_cl.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>